In [80]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings, 
    ChatGoogleGenerativeAI
)
from langchain_openai import (
    ChatOpenAI, 
    OpenAIEmbeddings
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(
    find_dotenv()
)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


data_path = "Data\\ReAct.pdf"
documents = PyMuPDFLoader(data_path).load()
print("Total pages found :- {}\n".format(len(documents)))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512, 
    chunk_overlap=64,
    length_function=len,
    is_separator_regex=True,
)

texts = splitter.split_documents(documents)
print("Total splitted documents chunks created are :- {}\n".format(len(texts)))


Total pages found :- 33

Total splitted documents chunks created are :- 250



In [ ]:
for index, text in enumerate(texts) :
    text.metadata["id"] = index


embeddings_google = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001"
)
embeddings_openai = OpenAIEmbeddings()


llm_model_google = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash-001", 
    temperature=0.4, 
    max_tokens=1024, 
    top_p=0.9
)

llm_model_openai = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.4, 
    max_tokens=1024, 
    top_p=0.9
)

retriever = FAISS.from_documents(
    texts, embeddings_google,
).as_retriever(
    search_type="similarity",
    search_kwargs={"k" : 10}
)


In [82]:
question = "Explain react algorithm as mentioned in the paper."
retrieved_documents = retriever.invoke(question)
print(retrieved_documents)

[Document(id='b0dd65f8-786a-417d-8887-7fcc6b447fac', metadata={'producer': 'macOS Version 13.4.1 (Build 22F82) Quartz PDFContext', 'creator': 'LaTeX with hyperref', 'creationdate': "D:20230901005855Z00'00'", 'source': 'Data\\ReAct.pdf', 'file_path': 'Data\\ReAct.pdf', 'total_pages': 33, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20230901005855Z00'00'", 'trapped': '', 'modDate': "D:20230901005855Z00'00'", 'creationDate': "D:20230901005855Z00'00'", 'page': 13, 'id': 119}, page_content='https://react-lm.github.io/.\nA.2\nREACT OBTAINS UP-TO-DATE KNOWLEDGE ON HOTPOTQA\n\x0b\x14D\x0c\x036WDQGDUG\n$QVZHU\x1d\x03\x16\x0f\x13\x13\x13\n\x0b\x14E\x0c\x03&R7\x03\x0b5HDVRQ\x032QO\\\x0c\n7KRXJKW\x1d\x03/HW\nV\x03WKLQN\x03VWHS \x03\nE\\\x03VWHS\x11\x037KH\x03KRWHO\x03WKDW\x03LV \x03\nKRPH\x03WR\x03WKH\x03&LUTXH\x03GX \x03\n6ROHLO\x03VKRZ\x030\\VWHUH\x03LV \x03\n7UHDVXUH\x03,VODQG\x11\x03 7UHDVXUH \x03\n,VODQG\x03KDV\x03\x15\x0f\x1b\x1b\x18\x03URRPV\x

In [83]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

compressor = FlashrankRerank()

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=retriever,
)

compressed_documents = compression_retriever.invoke(
    question
)
print("Reranked Documents are :- {}".format(
    [doc.metadata["id"] for doc in compressed_documents]
    )
)

Reranked Documents are :- [81, 181, 44]


In [84]:
from langchain.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(
    llm=llm_model_google, 
    retriever=compression_retriever, 
    chain_type="stuff",
)

In [85]:
final_reranked_answer = chain.invoke(
    question
)

In [86]:
print(final_reranked_answer["result"])

The provided text describes ReAct as an algorithm that integrates model actions and their corresponding observations into a coherent stream of inputs for the model to reason more accurately. 

Here's a breakdown based on the provided text:

* **ReAct goes beyond isolated reasoning:** Unlike traditional methods, ReAct doesn't just perform fixed reasoning steps. It actively interacts with the environment by taking actions and observing the results.
* **Action-Observation Loop:** ReAct operates in a loop where the model takes an action (e.g., searching for information online), observes the outcome of that action, and then uses this information to refine its reasoning process.
* **Coherent Input Stream:** The actions and observations are integrated into a continuous stream of inputs for the model, allowing it to build a more comprehensive understanding of the task and make more accurate decisions.
* **Task Beyond Reasoning:** ReAct can handle tasks that go beyond simple reasoning, such as 